# Kalendarz cech czasowych

Notebook generuje tabelę kalendarza (jeden wiersz = jeden dzień) z cechami czasowymi wykorzystywanymi przez modele. Wynik zapisywany jest do `dane/kalendarz/Kalendarz.csv` i dołączany do danych sprzedażowych.

### 1. Zakres dat
Utworzenie ciągłego zakresu dat od 01.01.2021 do 31.12.2025 (1826 dni).

In [13]:
import pandas as pd
import numpy as np
import holidays

In [14]:
# Daty 
daty = pd.date_range(
    start="2021-01-01",
    end="2025-12-31",
    freq="D"
)

df = pd.DataFrame({"data": daty})

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    1826 non-null   datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 14.4 KB


### 2. Cechy kalendarzowe i okresy specjalne
Funkcja `przygotuj_dane_pod_xgboost` dodaje:
* **Cechy podstawowe:** miesiąc, kwartał, dzień tygodnia (0 = poniedziałek), numer tygodnia (ISO), rok oraz flagę `czy_niedziela`.
* **Kodowanie cykliczne (sin/cos)** miesiąca, kwartału i dnia tygodnia – dzięki temu model „wie”, że po grudniu następuje styczeń, a po niedzieli poniedziałek.
* **`okres_wielkanocny`** – 14 dni przed Wielkanocą (do Wielkiej Soboty włącznie); data Wielkanocy wyliczana jest funkcją `easter`.
* **`okres_wigilijny`** – okres przedświątecznych zakupów: 10–23 grudnia.
* **`czy_majowka`** – okres zakupów przed majówką: 23–30 kwietnia.
* **`czy_ferie_slask`** – terminy ferii zimowych w województwie śląskim (wpisane ręcznie dla lat 2021–2026).
* **`czy_wakacje`** – lipiec i sierpień.

In [15]:
import pandas as pd
from dateutil.easter import easter
from datetime import timedelta

def przygotuj_dane_pod_xgboost(df):
    # 1. Konwersja daty
    df['data'] = pd.to_datetime(df['data'])
    
    # 2. Cechy kalendarzowe (XGBoost musi mieć je jako INT)
    df['miesiac'] = df['data'].dt.month
    df['kwartał'] = df['data'].dt.quarter
    # Dodajemy bo aby model wiedział że po 12 jest 1 
    df['miesiac_sin'] = np.sin(2 * np.pi * df['miesiac'] / 12)
    df['miesiac_cos'] = np.cos(2 * np.pi * df['miesiac'] / 12)
    df['kwartał_sin'] = np.sin(2 * np.pi * df['kwartał']/4)
    df['kwartał_cos'] = np.cos(2 * np.pi * df['kwartał']/4)
    df['dzien_tygodnia'] = df['data'].dt.dayofweek # 0=Pon, 6=Niedz
    # Przekształcenie żeby model wiedział że po niedzieli jest poniedziałek
    df['day_sin'] = np.sin(2 * np.pi * df['dzien_tygodnia'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['dzien_tygodnia'] / 7)
    df['tydzien_roku'] = df['data'].dt.isocalendar().week.astype(int)
    df['rok'] = df['data'].dt.year
    df['czy_niedziela'] = (df['data'].dt.dayofweek == 6).astype(int)

    # 3. Inicjalizacja flag
    df['okres_wielkanocny'] = 0
    df['okres_wigilijny'] = 0
    df['czy_majowka'] = 0
    lata = df['data'].dt.year.unique()

    for rok in lata:
        # WIELKANOC
        data_wielkanocy = easter(rok)
        poczatek_w = pd.to_datetime(data_wielkanocy - timedelta(days=14))
        koniec_w = pd.to_datetime(data_wielkanocy - timedelta(days=1))
        df.loc[df['data'].between(poczatek_w, koniec_w), 'okres_wielkanocny'] = 1

        # WIGILIA
        poczatek_x = pd.to_datetime(f"{rok}-12-10")
        koniec_x = pd.to_datetime(f"{rok}-12-23")
        df.loc[df['data'].between(poczatek_x, koniec_x), 'okres_wigilijny'] = 1
        
        # Majówka: 1-3 maja
 
        poczatek_x = pd.to_datetime(f"{rok}-04-23")
        koniec_x = pd.to_datetime(f"{rok}-04-30")
        df.loc[df['data'].between(poczatek_x, koniec_x), 'czy_majowka'] = 1    
        
    ferie_slask = {
        2021: ('2021-01-04', '2021-01-17'),
        2022: ('2022-02-14', '2022-02-27'),
        2023: ('2023-01-16', '2023-01-29'),
        2024: ('2024-01-29', '2024-02-11'),
        2025: ('2025-02-17', '2025-03-02'),
        2026: ('2026-02-02', '2026-02-15')
    }
    df['czy_ferie_slask'] = 0
    for rok, (start, end) in ferie_slask.items():
        maska = df['data'].between(start, end)
        df.loc[maska, 'czy_ferie_slask'] = 1

    # Wakacje: lipiec i sierpień
    df['czy_wakacje'] = df['data'].dt.month.isin([7, 8]).astype(int)

    return df

df = przygotuj_dane_pod_xgboost(df)

### 3. Święta ustawowe
Nazwy świąt państwowych w Polsce pobierane są z biblioteki `holidays` (kolumna `nazwa_swieta`, dla pozostałych dni `Brak`). Dodatkowo Wigilia (24 grudnia) oznaczona jest jako święto. Na tej podstawie tworzona jest flaga `czy_swieto`.

In [16]:
pl_holidays = holidays.Poland()

df['nazwa_swieta'] = df['data'].map(
    lambda x: pl_holidays.get(x, 'Brak'))

df['czy_swieto'] = (df['nazwa_swieta'] != 'Brak').astype(int)
# dodanie wigili jako święto 
df.loc[
    (df['data'].dt.month == 12) & (df['data'].dt.day == 24),
    'nazwa_swieta'
] = 'Wigilia'
df['czy_swieto'] = (df['nazwa_swieta'] != 'Brak').astype(int)


### 4. Liczba dni do najbliższego święta
Dla każdego dnia liczona jest liczba dni do najbliższego święta (`dni_do_swieta`; w dniu święta = 0). Zakres dat kończy się 31.12.2025, więc dla ostatnich dni grudnia 2025 kolejne święto nie występuje w danych, wartości dla 27–31.12.2025 uzupełniono ręcznie, licząc dni do Nowego Roku 2026.

In [17]:
swieta = df.loc[df['czy_swieto'] == 1, 'data'].values

def dni_do_swieta(data):
    przyszle = swieta[swieta >= data]
    if len(przyszle) == 0:
        return 999
    return (przyszle.min() - data).days

df['dni_do_swieta'] = df['data'].apply(dni_do_swieta)

### 5. Kontrola poprawności
Podgląd wybranych dni oraz struktury tabeli (1826 wierszy, 21 kolumn).

In [18]:
df[df['data'] =='2021-10-23']

,data,miesiac,kwartał,miesiac_sin,miesiac_cos,kwartał_sin,kwartał_cos,dzien_tygodnia,day_sin,day_cos,...,rok,czy_niedziela,okres_wielkanocny,okres_wigilijny,czy_majowka,czy_ferie_slask,czy_wakacje,nazwa_swieta,czy_swieto,dni_do_swieta
295,2021-10-23,10,4,-0.866025,0.5,-2.449294e-16,1.0,5,-0.974928,-0.222521,...,2021,0,0,0,0,0,0,Brak,0,9


In [19]:
df.loc[df['data'] == '2025-12-27', 'dni_do_swieta'] = 5
df.loc[df['data'] == '2025-12-28', 'dni_do_swieta'] = 4
df.loc[df['data'] == '2025-12-29', 'dni_do_swieta'] = 3
df.loc[df['data'] == '2025-12-30', 'dni_do_swieta'] = 2
df.loc[df['data'] == '2025-12-31', 'dni_do_swieta'] = 1

In [20]:
df.tail(10)

,data,miesiac,kwartał,miesiac_sin,miesiac_cos,kwartał_sin,kwartał_cos,dzien_tygodnia,day_sin,day_cos,...,rok,czy_niedziela,okres_wielkanocny,okres_wigilijny,czy_majowka,czy_ferie_slask,czy_wakacje,nazwa_swieta,czy_swieto,dni_do_swieta
1816,2025-12-22,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,0,0.000000,1.000000,...,2025,0,0,1,0,0,0,Brak,0,2
1817,2025-12-23,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,1,0.781831,0.623490,...,2025,0,0,1,0,0,0,Brak,0,1
1818,2025-12-24,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,2,0.974928,-0.222521,...,2025,0,0,0,0,0,0,Wigilia,1,0
1819,2025-12-25,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,3,0.433884,-0.900969,...,2025,0,0,0,0,0,0,Boże Narodzenie (pierwszy dzień),1,0
1820,2025-12-26,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,4,-0.433884,-0.900969,...,2025,0,0,0,0,0,0,Boże Narodzenie (drugi dzień),1,0
1821,2025-12-27,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,5,-0.974928,-0.222521,...,2025,0,0,0,0,0,0,Brak,0,5
1822,2025-12-28,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,6,-0.781831,0.623490,...,2025,1,0,0,0,0,0,Brak,0,4
1823,2025-12-29,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,0,0.000000,1.000000,...,2025,0,0,0,0,0,0,Brak,0,3
1824,2025-12-30,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,1,0.781831,0.623490,...,2025,0,0,0,0,0,0,Brak,0,2
1825,2025-12-31,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,2,0.974928,-0.222521,...,2025,0,0,0,0,0,0,Brak,0,1


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   data               1826 non-null   datetime64[ns]
 1   miesiac            1826 non-null   int32         
 2   kwartał            1826 non-null   int32         
 3   miesiac_sin        1826 non-null   float64       
 4   miesiac_cos        1826 non-null   float64       
 5   kwartał_sin        1826 non-null   float64       
 6   kwartał_cos        1826 non-null   float64       
 7   dzien_tygodnia     1826 non-null   int32         
 8   day_sin            1826 non-null   float64       
 9   day_cos            1826 non-null   float64       
 10  tydzien_roku       1826 non-null   int64         
 11  rok                1826 non-null   int32         
 12  czy_niedziela      1826 non-null   int64         
 13  okres_wielkanocny  1826 non-null   int64         
 14  okres_wi

In [22]:
df[df['data'] == '2021-12-12']

,data,miesiac,kwartał,miesiac_sin,miesiac_cos,kwartał_sin,kwartał_cos,dzien_tygodnia,day_sin,day_cos,...,rok,czy_niedziela,okres_wielkanocny,okres_wigilijny,czy_majowka,czy_ferie_slask,czy_wakacje,nazwa_swieta,czy_swieto,dni_do_swieta
345,2021-12-12,12,4,-2.449294e-16,1.0,-2.449294e-16,1.0,6,-0.781831,0.62349,...,2021,1,0,1,0,0,0,Brak,0,12


### 6. Zapis kalendarza
Zapis do `../dane/kalendarz/Kalendarz.csv` (separator `;`, przecinek jako separator dziesiętny).

In [23]:
df.to_csv(
    '../dane/kalendarz/Kalendarz.csv', 
    encoding='utf-8-sig', 
    sep=';',       # Rozdzielacz kolumn (średnik)
    decimal=',',   # <--- TO JEST KLUCZ! Zamienia kropki na przecinki w liczbach
    index=False
)